In [ ]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("spam-experiment")

In [ ]:
# from spam_checker.data.spam_data_module import SPAMDataModule

In [ ]:
# spam_data = SPAMDataModule()

In [ ]:
# spam_data.prepare_data()

In [ ]:
import pandas as pd

# Paste your copied file path inside the quotes
# df = pd.read_csv('../test_dataset/spam.csv')

df = pd.read_csv('../test_dataset/spam.csv', encoding_errors='ignore')

df = df[['v1', 'v2']]

# df['v1'] = df['v1'].replace('spam', '1')
# df['v1'] = df['v1'].replace('ham', '0')
df = df.rename(columns={"v1": "label", "v2": "text"})

# View the first few rows of your data
df.head(10)

In [ ]:
# df['label'] = df['label'].replace('spam', '1')
# df['label'] = df['label'].replace('ham', '0')

# # View the first few rows of your data
# df.head(10)

In [ ]:
len(df.index)

In [ ]:
empty_spam_rows = df[df['text'].isna()]
print(empty_spam_rows)

In [ ]:
empty_count = df['text'].isna().sum()
print(f"Empty cells: {empty_count}")

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [ ]:
doc = nlp("spaCy is successfully working in Jupyter!")
for token in doc:
    print(token.text, token.pos_)

In [ ]:
# Function to clean the text
def clean_text(text):
    doc = nlp(text)  # Process the text with spaCy
    cleaned_tokens = []
    for token in doc:
        # Remove stop words, punctuation, and retain only alphabetic words
        if not token.is_stop and not token.is_punct and token.is_alpha:
            cleaned_tokens.append(token.lemma_)  # Append the lemmatized version of the token
    return " ".join(cleaned_tokens)

# Apply the cleaning function to the 'text' column in the dataset
# df['cleaned_text'] = df['text'].apply(clean_text)

In [ ]:
df.head(10)

In [ ]:
training_input_labels = df.loc[:, df.columns == 'label']
training_input_text = df.loc[:, df.columns != 'label']

print(training_input_labels['label'].value_counts())

In [ ]:
print(training_input_labels)

In [ ]:
from sklearn.model_selection import train_test_split

train_text_temp, val_text_final, train_labels_temp, val_labels = train_test_split(
    training_input_text, training_input_labels, train_size=0.5, random_state=0, stratify=training_input_labels)

train_text_final, test_text_final, train_labels, test_labels = train_test_split(
    train_text_temp, train_labels_temp, train_size=0.8, random_state=0, stratify=train_labels_temp)


In [ ]:
train_text_combined = pd.concat([train_labels, train_text_final], axis=1).reset_index(drop=True)

In [ ]:
test_text_combined = pd.concat([test_labels, test_text_final], axis=1).reset_index(drop=True)

In [ ]:
val_text_combined = pd.concat([val_labels, val_text_final], axis=1).reset_index(drop=True)

In [ ]:
empty_spam_rows = train_text_combined[train_text_combined['text'].isna()]
print(empty_spam_rows)

In [ ]:
# empty_spam_rows = train_text_combined[train_text_combined['cleaned_text'].isna()]
# print(empty_spam_rows)

In [ ]:
empty_spam_rows = train_text_combined[train_text_combined['label'].isna()]
print(empty_spam_rows)

In [ ]:
def spacy_tokenizer(text):
    # Use the clean_lemm function first
    cleaned_text = clean_text(text)
    # Then tokenize the cleaned text
    return cleaned_text.split()

In [ ]:
from collections import defaultdict

def build_vocab(text_iterator, min_freq=3, specials=('<unk>', '<pad>')):
    token_counts = defaultdict(int)
    for text in text_iterator:
        for token in spacy_tokenizer(text):
            token_counts[token] += 1
    vocab = {token: idx for idx, (token, count) in enumerate(token_counts.items()) if count >= min_freq}
    for special in specials:
        if special not in vocab:
            vocab[special] = len(vocab) 
    return vocab
vocab = build_vocab(df['text'])
# vocab = build_vocab(df['cleaned_text'])
vocab_size = len(vocab)

In [ ]:
print(vocab_size)

In [ ]:
print(vocab)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

# 1. Mock Data (Replace this with your SMS Spam Collection CSV file)
# data = [
#     ("Go jurong point, crazy.. Available only in bugis n great world la e buffet...", "ham"),
#     ("Ok lar... Joking wif u oni...", "ham"),
#     ("Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005.", "spam"),
#     ("U dun say so early hor... U c already then say...", "ham"),
#     ("FreeMsg Hey there darling it's been 3 week's now no word back!", "spam"),
#     ("WINNER!! As a valued network customer you have been selected to receivea £900 prize reward!", "spam"),
#     ("Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free!", "spam"),
#     ("I'm gonna be home soon and i don't want to talk about this stuff anymore tonight, k?", "ham"),
# ]

data = df
 

# 2. Text Preprocessing and Vocabulary Builder
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

# Build Vocabulary
all_words = []
for text in data['text']:
    all_words.extend(clean_text(text))

word_counts = Counter(all_words)
# Filter words that appear at least once; add <PAD> and <UNK> tokens
vocab = {"<PAD>": 0, "<UNK>": 1}
for word in word_counts:
    if word not in vocab:
        vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)
MAX_LEN = 20  # Max sequence length for padding

print(vocab)

In [ ]:
# 3. Custom PyTorch Dataset
class SMSDataset(Dataset):
    def __init__(self, data, vocab, max_len):
        self.data = data['text']
        self.labels = data['label']
        self.vocab = vocab
        self.max_len = max_len
        self.labels_map = {"ham": 0, "spam": 1}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]
        label = self.labels[idx]
        words = clean_text(text)
        
        # Numericalize text (Word -> ID)
        encoded = [self.vocab.get(word, self.vocab["<UNK>"]) for word in words]
        
        # Pad or truncate sequence to fixed length
        if len(encoded) < self.max_len:
            encoded += [self.vocab["<PAD>"]] * (self.max_len - len(encoded))
        else:
            encoded = encoded[:self.max_len]
            
        return torch.tensor(encoded, dtype=torch.long), torch.tensor(self.labels_map[label], dtype=torch.float)

# Create Dataset and DataLoader
dataset = SMSDataset(data, vocab, MAX_LEN)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)



In [ ]:
# 4. Simple RNN Classification Model
class SpamClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SpamClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # x shape: [batch_size, max_len]
        embedded = self.embedding(x) # [batch_size, max_len, embedding_dim]
        lstm_out, (hidden, cell) = self.lstm(embedded) # hidden shape: [1, batch_size, hidden_dim]
        
        # Use the final hidden state of the LSTM
        last_hidden = hidden.squeeze(0) # [batch_size, hidden_dim]
        out = self.fc(last_hidden) # [batch_size, 1]
        return self.sigmoid(out)

# Hyperparameters
EMBEDDING_DIM = 32
HIDDEN_DIM = 16

model = SpamClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 5. Training Loop
print("Starting Training...")
model.train()
for epoch in range(5):
    epoch_loss = 0
    for texts, labels in dataloader:
        optimizer.zero_grad()
        
        predictions = model(texts).squeeze(1) # shape matches labels [batch_size]
        loss = criterion(predictions, labels)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1} | Loss: {epoch_loss / len(dataloader):.4f}")

# 6. Inference Example
def predict(text):
    model.eval()
    with torch.no_grad():
        words = clean_text(text)
        encoded = [vocab.get(word, vocab["<UNK>"]) for word in words]
        if len(encoded) < MAX_LEN:
            encoded += [vocab["<PAD>"]] * (MAX_LEN - len(encoded))
        else:
            encoded = encoded[:MAX_LEN]
            
        test_tensor = torch.tensor([encoded], dtype=torch.long)
        prediction = model(test_tensor).item()
        return "Spam" if prediction > 0.5 else "Ham"

print("\nTesting Predictions:")
print(f"'Free entry to win cash now!' -> {predict('Free entry to win cash now!')}")
print(f"'Hey, are we still meeting for lunch?' -> {predict('Hey, are we still meeting for lunch?')}")